# D1 · Parco fotovoltaico: dalla previsione meteo all'offerta di energia (C#)

Stessi numeri nel notebook [PySharp](D1-fotovoltaico-pysharp.ipynb) (facciata `Interop`); un test verifica che coincidano. Kernel JupyterNet su .NET 11.

## Il caso

Un **parco fotovoltaico da 1 MWp** (inverter da 600 kW AC) vende energia sul mercato del giorno prima. Il SCADA registra ogni ora irraggiamento, temperature, potenza AC e stato dell'inverter; un servizio meteo manda, il giorno prima, la **previsione di irraggiamento** per le 24 ore successive, con il suo errore.

**Chi lavora sui dati.** L'*energy manager* deve presentare l'offerta di energia di domani; chi fa *O&M* vuole sapere se l'impianto rende come dovrebbe (e se vale la pena lavare i moduli); la *proprietà* vuole sapere perché un giorno si è prodotto meno. La domanda del cliente (il trader): **«quanta energia offrite sul mercato domani?»**, e va data **con un'incertezza**, perché produrre meno di quanto offerto costa di più che produrre di più.

**Dati sintetici** (`generate-data.cs`, con seme): 8 settimane orarie. Le prime 7 per addestrare, l'ultima come prova mai vista. La verità nota è in `data/verita.json`: la pulizia dei moduli (giorno 30), un guasto dell'inverter (giorno 38, 5 ore) e l'errore del servizio meteo.

| Passo | Domanda | Strumento |
|---|---|---|
| 1 | L'impianto rende come dovrebbe? | KPI: performance ratio, disponibilità |
| 2 | Quanta potenza darà domani, **sapendo il meteo previsto**? | regressione sulla covariata, seasonal naive, MASE |
| 3 | Con quale incertezza? | quantili dei residui per ora, perdita pinball, copertura |
| 4 | E sull'energia dell'**intera giornata**? | traiettorie campionate (blocchi di residui) |
| 5 | Quanta energia offro? | quantile ottimo dato il costo dell'errore |
| 6 | KPI, dashboard, risposta al cliente | tile, grafici, testo con i numeri |

In [ ]:
#load "../../../shared/MathemaSharp.csx"
using MathemaSharp.TimeSeries;
using System.Globalization;
using System.IO;
using System.Runtime.CompilerServices;

static void P(string label, IEnumerable<double> values) =>
    Console.WriteLine($"{label}: {string.Join(" ", values.Select(v => v.ToString("F6", CultureInfo.InvariantCulture)))}");
static void P(string label, double value) => P(label, new[] { value });
static string Here([CallerFilePath] string path = "") => Path.GetDirectoryName(path)!;   // cartella di questo notebook
static double Num(string s) => double.Parse(s, CultureInfo.InvariantCulture);

var data = Path.Combine(Here(), "data");
var rows = File.ReadLines(Path.Combine(data, "impianto_orario.csv")).Skip(1).Select(l => l.Split(',')).ToArray();
double[] G = rows.Select(r => Num(r[1])).ToArray(), Gf = rows.Select(r => Num(r[2])).ToArray(), Tair = rows.Select(r => Num(r[3])).ToArray();
double[] Pac = rows.Select(r => Num(r[5])).ToArray(), ok = rows.Select(r => Num(r[6])).ToArray();
const int trainDays = 49, trainHours = trainDays * 24, total = 56 * 24;
const double pMax = 600, pNom = 1000;   // kW AC massimi dell'inverter, kWp installati

## 1 · Acquisire e misurare: l'impianto rende come dovrebbe?
Il **performance ratio** (IEC 61724) è l'energia prodotta divisa per quella che produrrebbe un impianto ideale con la stessa insolazione: $PR=E_{AC}/(H_{POA}\cdot P_{nom})$. Misura la salute dell'impianto *a prescindere dal meteo*. La **disponibilità** conta le ore di sole in cui l'inverter era in servizio.

In [ ]:
var dailyEnergy = new double[56];
var dailyInsolation = new double[56];
var pr = new double[56];
for (var d = 0; d < 56; d++)
{
    for (var h = 0; h < 24; h++) { dailyEnergy[d] += Pac[d * 24 + h]; dailyInsolation[d] += G[d * 24 + h] / 1000; }
    pr[d] = dailyInsolation[d] > 0.5 ? dailyEnergy[d] / (dailyInsolation[d] * pNom) : double.NaN;
}
double MeanPr(int from, int to) => Enumerable.Range(from, to - from).Select(d => pr[d]).Average();
P("pr_prima_pulizia", MeanPr(24, 30));
P("pr_dopo_pulizia", MeanPr(31, 37));
P("pr_giorno_guasto", pr[38]);
var sunny = Enumerable.Range(0, total).Where(i => G[i] > 50).ToArray();
P("disponibilita", sunny.Count(i => ok[i] == 1) / (double)sunny.Length);
P("ore_al_limite_inverter", Pac.Count(p => p >= pMax - 0.05));

Il PR sale dopo la pulizia dei moduli (giorno 30) e precipita nel giorno del guasto: il KPI vede entrambi. Ma la differenza dovuta alla pulizia è di pochi punti: serve una **media su più giorni** per distinguerla dal rumore.

In [ ]:
Chart.Lines("Performance ratio giornaliero (pulizia al giorno 30, guasto al 38)", new Line("PR", pr));

## 2 · Prevedere la potenza di domani, sapendo il meteo previsto
La potenza dipende dall'irraggiamento e un servizio meteo lo prevede il giorno prima: è la **covariata nota nel futuro** (con il suo errore). Modello: regressione $P\approx\beta_0+\beta_1G_{prev}+\beta_2G_{prev}\,T_{aria}$, limitata a 0…600 kW, addestrata a QR sulle ore in cui l'inverter era in servizio. Riferimento: il **seasonal naive a un giorno** (la stessa ora di ieri).

In [ ]:
var trainRows = Enumerable.Range(0, trainHours).Where(i => ok[i] == 1).ToArray();   // fuori le ore di guasto: non sono "normali"
var X = new double[trainRows.Length, 3];
var y = new double[trainRows.Length];
for (var r = 0; r < trainRows.Length; r++)
{
    var i = trainRows[r];
    X[r, 0] = 1; X[r, 1] = Gf[i]; X[r, 2] = Gf[i] * Tair[i];
    y[r] = Pac[i];
}
var beta = MathemaSharp.LinearAlgebra.MatrixAlgebra.LeastSquares<double>(X, y);
P("beta", beta);
double Predict(int i) => Gf[i] <= 0 ? 0 : Math.Clamp(beta[0] + beta[1] * Gf[i] + beta[2] * Gf[i] * Tair[i], 0, pMax);

double[] train = Pac[..trainHours], test = Pac[trainHours..];
var model = Enumerable.Range(trainHours, 168).Select(Predict).ToArray();
var persistence = Pac[(trainHours - 24)..(total - 24)];                              // ieri alla stessa ora
P("mase_modello", Accuracy.Mase<double>(test, model, train, 24));
P("mase_persistenza", Accuracy.Mase<double>(test, persistence, train, 24));
try { Accuracy.Mape<double>(test, model); P("mape_definita", 1); } catch (ArgumentException) { P("mape_definita", 0); }   // di notte la potenza è 0

**La MAPE non si può calcolare**: di notte la potenza vale zero e la libreria lo dichiara invece di restituire un numero senza senso. È il motivo per cui in questo libro si usa la **MASE**: definita anche con gli zeri e indipendente dalla scala.

## 3 · Con quale incertezza? Quantili dei residui per ora del giorno
Le previsioni sbagliano in modo diverso a ogni ora (a mezzogiorno l'errore è grande, di notte nullo). Si prendono i **residui in campione** del modello (potenza reale − prevista) **ora per ora** e se ne calcolano i quantili: sono gli scarti da sommare alla previsione per ottenere P10, P50, P90.

In [ ]:
var residual = new double[trainHours];
for (var i = 0; i < trainHours; i++) residual[i] = Pac[i] - Predict(i);
double[] HourQuantile(int hour, double p) => new[] { SeriesStatistics.Quantile<double>(trainRows.Where(i => i % 24 == hour).Select(i => residual[i]).ToArray(), p) };
var q10 = Enumerable.Range(0, 24).Select(h => HourQuantile(h, 0.1)[0]).ToArray();
var q50 = Enumerable.Range(0, 24).Select(h => HourQuantile(h, 0.5)[0]).ToArray();
var q90 = Enumerable.Range(0, 24).Select(h => HourQuantile(h, 0.9)[0]).ToArray();

double Band(int i, double[] q) => Gf[i] <= 0 ? 0 : Math.Clamp(Predict(i) + q[i % 24], 0, pMax);
var lo = Enumerable.Range(trainHours, 168).Select(i => Band(i, q10)).ToArray();
var mid = Enumerable.Range(trainHours, 168).Select(i => Band(i, q50)).ToArray();
var hi = Enumerable.Range(trainHours, 168).Select(i => Band(i, q90)).ToArray();
P("pinball_p10", Accuracy.PinballLoss<double>(test, lo, 0.1));
P("pinball_p50", Accuracy.PinballLoss<double>(test, mid, 0.5));
P("pinball_p90", Accuracy.PinballLoss<double>(test, hi, 0.9));
P("copertura_p10_p90_tutte_le_ore", Accuracy.IntervalCoverage<double>(test, lo, hi));
// Di notte la potenza è 0 e l'intervallo è [0, 0]: contiene sempre il valore e gonfia la copertura. Si misura solo sulle ore di sole.
var daylight = Enumerable.Range(0, 168).Where(t => Gf[trainHours + t] > 0).ToArray();
double[] At(double[] x) => daylight.Select(t => x[t]).ToArray();
var coverageDay = Accuracy.IntervalCoverage<double>(At(test), At(lo), At(hi));
P("copertura_p10_p90_ore_di_sole", coverageDay);

L'intervallo P10–P90 dovrebbe contenere l'**80 %** delle osservazioni. Contando **tutte** le ore la copertura sembra alta, ma è un'illusione: di notte la potenza è zero e l'intervallo è [0, 0], quindi contiene sempre il valore. Misurata **solo sulle ore di sole** è il numero che conta; se risultasse più bassa dell'80 %, l'intervallo sarebbe troppo ottimista (i residui in campione sottostimano l'errore su dati nuovi).

In [ ]:
int a = 96, b = 168;   // gli ultimi 3 giorni della prova
Chart.Lines("Ultimi 3 giorni: potenza reale e fascia P10–P90 (kW, asse x: ore)",
    new Line("reale", test[a..b]), new Line("P50", mid[a..b], Dashed: true), new Line("P10", lo[a..b], Dashed: true), new Line("P90", hi[a..b], Dashed: true));

## 4 · E l'energia dell'**intera giornata**?
La domanda del trader riguarda i kWh del giorno, non la potenza di un'ora. **Sommare i P10 orari non dà il P10 del giorno**: equivale a supporre che tutte le 24 ore sbaglino insieme nel peggiore dei modi. Si preservano invece le **dipendenze nel tempo** campionando traiettorie: a ogni giornata di addestramento si prende il suo intero vettore di residui, lo si somma alla previsione e si ottiene un'energia possibile. Dalle 48 energie possibili si prendono i quantili (giorno del guasto escluso).

In [ ]:
var okDays = Enumerable.Range(0, trainDays).Where(d => d != 38).ToArray();
var p10Day = new double[7]; var p50Day = new double[7]; var p90Day = new double[7]; var p25Day = new double[7];
var actualDay = new double[7]; var sumLo = new double[7]; var sumHi = new double[7];
var trajectoryTotals = new double[7][];
for (var d = 0; d < 7; d++)
{
    var totals = new double[okDays.Length];
    for (var k = 0; k < okDays.Length; k++)
    {
        double s = 0;
        for (var h = 0; h < 24; h++)
        {
            var i = trainHours + d * 24 + h;
            s += Gf[i] <= 0 ? 0 : Math.Clamp(model[d * 24 + h] + residual[okDays[k] * 24 + h], 0, pMax);
        }
        totals[k] = s;
    }
    trajectoryTotals[d] = totals;
    p10Day[d] = SeriesStatistics.Quantile<double>(totals, 0.1);
    p25Day[d] = SeriesStatistics.Quantile<double>(totals, 0.25);
    p50Day[d] = SeriesStatistics.Quantile<double>(totals, 0.5);
    p90Day[d] = SeriesStatistics.Quantile<double>(totals, 0.9);
    for (var h = 0; h < 24; h++) { actualDay[d] += test[d * 24 + h]; sumLo[d] += lo[d * 24 + h]; sumHi[d] += hi[d * 24 + h]; }
}
P("energia_reale", actualDay);
P("energia_p10", p10Day);
P("energia_p50", p50Day);
P("energia_p90", p90Day);
P("energia_somma_p10", sumLo);
P("energia_somma_p90", sumHi);
int Inside(double[] low, double[] high) => Enumerable.Range(0, 7).Count(d => actualDay[d] >= low[d] && actualDay[d] <= high[d]);
P("giorni_dentro_traiettorie", Inside(p10Day, p90Day));
P("giorni_dentro_somma_oraria", Inside(sumLo, sumHi));
P("larghezza_traiettorie_kwh", Enumerable.Range(0, 7).Average(d => p90Day[d] - p10Day[d]));
P("larghezza_somma_oraria_kwh", Enumerable.Range(0, 7).Average(d => sumHi[d] - sumLo[d]));

La banda **dalle traiettorie** è più stretta di quella ottenuta sommando i quantili orari (circa **1.5 MWh** contro **1.9 MWh** in media): tiene conto di quanto gli errori si compensano durante il giorno, mentre la somma assume il caso peggiore su tutte le ore insieme, ed è per questo che contiene sempre tutti e sette i giorni. I giorni dentro la banda delle traiettorie sono **4 su 7** contro l'80 % atteso: con soli 7 giorni è compatibile con la sorte (vedere 4 o meno su 7 con probabilità 0.8 capita circa 15 volte su 100). **Non è una calibrazione**: ci vogliono molte più origini di prova.

## 5 · Quanta energia offrire? Il quantile che costa meno
Sul mercato, **produrre meno di quanto offerto costa di più** che produrre di più: se lo scarto per difetto costa $a$ per kWh e quello per eccesso $b$, l'offerta che minimizza il costo atteso è il **quantile di livello $b/(a+b)$** della distribuzione dell'energia: è la perdita pinball del capitolo sulle previsioni probabilistiche. Con $a=3,\;b=1$ il livello è **0.25**.

In [ ]:
double Cost(double offer, double actual) => 3 * Math.Max(offer - actual, 0) + 1 * Math.Max(actual - offer, 0);   // a = 3 (difetto), b = 1 (eccesso)
double WeekCost(double[] offers) => Enumerable.Range(0, 7).Sum(d => Cost(offers[d], actualDay[d]));
P("costo_offerta_p10", WeekCost(p10Day));
P("costo_offerta_p25", WeekCost(p25Day));
P("costo_offerta_p50", WeekCost(p50Day));
P("costo_offerta_p90", WeekCost(p90Day));
var bestOffer = new[] { ("p10", p10Day), ("p25", p25Day), ("p50", p50Day), ("p90", p90Day) }.OrderBy(o => WeekCost(o.Item2)).First();
Console.WriteLine($"offerta_migliore: {bestOffer.Item1}");

Sulla settimana di prova il livello 0.25 costa meno di offrire la mediana (P50) e molto meno del P90, come la teoria dice. Con sette giorni è una conferma, non una dimostrazione: la teoria vale **in media**, e il costo di una settimana è rumoroso.

## 6 · Comunicare: KPI, dashboard e risposta al cliente

In [ ]:
var lastDay = 6;
Kpi.Tiles(
    new Tile("Performance ratio", $"{MeanPr(31, 37):F3}", $"dopo la pulizia, contro {MeanPr(24, 30):F3} prima", "ok"),
    new Tile("Disponibilità", $"{100.0 * sunny.Count(i => ok[i] == 1) / sunny.Length:F1} %", "ore di sole con inverter in servizio; 5 ore di guasto il giorno 38", "ok"),
    new Tile("Energia domani (P25)", $"{p25Day[lastDay]:F0} kWh", $"P10 {p10Day[lastDay]:F0} – P90 {p90Day[lastDay]:F0} kWh; mediana {p50Day[lastDay]:F0}", "ok"),
    new Tile("Copertura P10–P90", $"{100 * coverageDay:F0} %", "ore di sole, sulla settimana di prova (atteso 80 %)", coverageDay < 0.7 ? "warn" : "ok"),
    new Tile("MASE del modello", $"{Accuracy.Mase<double>(test, model, train, 24):F2}", $"contro {Accuracy.Mase<double>(test, persistence, train, 24):F2} della persistenza (ieri = oggi)", "ok"));

In [ ]:
var answer = $"Per il 20 settembre offriamo {p25Day[lastDay]:F0} kWh. È il livello 25 % della distribuzione dell'energia attesa (mediana {p50Day[lastDay]:F0} kWh, intervallo 10–90 % da {p10Day[lastDay]:F0} a {p90Day[lastDay]:F0} kWh): " +
             $"il mercato penalizza il difetto tre volte l'eccesso, e offrire il livello {100 * 1.0 / 4:F0} % è ciò che minimizza il costo atteso. " +
             $"La previsione si basa sull'irraggiamento previsto dal servizio meteo: se il meteo cambia, aggiorneremo l'offerta entro la chiusura del mercato.";
Kpi.Quote("Risposta al cliente: «quanta energia offrite sul mercato domani?»", answer);

var oem = $"Il performance ratio medio è salito da {MeanPr(24, 30):F3} a {MeanPr(31, 37):F3} dopo la pulizia dei moduli del giorno 30: lo sporco costava circa il {100 * (MeanPr(31, 37) - MeanPr(24, 30)) / MeanPr(31, 37):F1} % di energia. Il giorno del guasto (38) il PR è sceso a {pr[38]:F2}: 5 ore di inverter fermo.";
Kpi.Quote("Nota per O&M", oem);

## Cosa abbiamo imparato
- **Il KPI giusto toglie il meteo**: il PR separa la salute dell'impianto dal tempo che fa.
- **Una covariata nota migliora molto** la previsione, ma porta con sé il suo errore: l'incertezza della previsione meteo *è* l'incertezza dell'offerta.
- **MAPE no, MASE sì**: con gli zeri notturni la MAPE non esiste.
- **I quantili orari non si sommano**: per l'energia del giorno si campionano traiettorie, preservando le dipendenze nel tempo.
- **Il costo dell'errore decide il quantile**: offrire la mediana non è ottimo quando il difetto costa più dell'eccesso.
- **Una settimana di prova non basta** per calibrare l'intervallo: la validazione su più origini (*rolling origin*) e la previsione conforme sono nel libro, non ancora nella libreria.